# FPSA-Prime verification and controlled Maze experiment

This notebook validates the numerical implementation first, then optionally runs the first architecture-level Maze comparison.

The full experiment keeps strict fixed-point checks enabled. Training uses a bounded recurrent budget, while evaluation gets a larger explicit ceiling so a small tail of hard examples is allowed to finish. The solver still stops adaptively, and the JSON output reports realized mean and tail iterations.


In [ ]:
REPO_URL = "https://github.com/mrinal18/fpsa.git"
BRANCH = "fpsa-prime-v0"

QUICK = True
RUN_TINY_TRAINING = True

RUN_FULL_BENCHMARK = False
FULL_SEEDS = [0]              # expand to [0, 1, 2] after one clean run
FULL_STEPS = 1200
TRAIN_MAX_ITER = 24
EVAL_MAX_ITER = 64


## 1. Checkout the latest branch and install dependencies


In [ ]:
from pathlib import Path
import json, os, shlex, subprocess, sys

IN_COLAB = "google.colab" in sys.modules
target = Path("/content/fpsa") if IN_COLAB else Path.cwd()

if not (target / ".git").exists():
    if target.exists() and target != Path.cwd():
        subprocess.run(["rm", "-rf", str(target)], check=True)
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(target)],
        check=True,
    )
else:
    os.chdir(target)
    subprocess.run(["git", "fetch", "origin", BRANCH], check=True)
    subprocess.run(
        ["git", "checkout", "-B", BRANCH, f"origin/{BRANCH}"],
        check=True,
    )
    subprocess.run(
        ["git", "reset", "--hard", f"origin/{BRANCH}"],
        check=True,
    )

os.chdir(target)
sys.path.insert(0, str(target))

if IN_COLAB:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "pytest",
            "pandas",
            "matplotlib",
        ],
        check=True,
    )

import pandas as pd
import matplotlib.pyplot as plt
import torch

print(
    {
        "git_sha": subprocess.check_output(
            ["git", "rev-parse", "HEAD"], text=True
        ).strip(),
        "torch": torch.__version__,
        "cuda": torch.cuda.is_available(),
        "gpu": (
            torch.cuda.get_device_name(0)
            if torch.cuda.is_available()
            else None
        ),
    }
)


## 2. Run the portable numerical verification harness


In [ ]:
# Remove stale modules if this cell is rerun after a git update.
for module_name in list(sys.modules):
    if module_name.startswith("experiments.fpsa_prime.verification"):
        del sys.modules[module_name]

from experiments.fpsa_prime.verification import run_verification

summary = run_verification(
    output_dir="results/verification_colab",
    device="auto",
    quick=QUICK,
    run_tests=True,
    run_training=RUN_TINY_TRAINING,
)

print(
    json.dumps(
        {
            "tests": summary["tests"].splitlines()[-1],
            "parameter_gap_percent": (
                100 * summary["hero_block_parameter_gap_fraction"]
            ),
            "fixed_unroll": summary["fixed_unroll"],
            "gmres_diagnostic": summary["gmres_diagnostic"],
            "plots": summary["plots"],
        },
        indent=2,
    )
)


## 3. Inspect verification tables


In [ ]:
parameter_table = pd.DataFrame(summary["parameter_counts"])
gmres_table = pd.DataFrame(summary["gmres"])
stability_table = pd.DataFrame(summary["stability_power_sweep"])

display(parameter_table)
display(gmres_table)
display(stability_table)

if summary["tiny_training"] is not None:
    display(pd.DataFrame(summary["tiny_training"]))


## 4. Inspect verification plots


In [ ]:
from IPython.display import Image, display

for plot_name in summary["plots"]:
    print(plot_name)
    display(Image(filename=f"results/verification_colab/{plot_name}"))


## 5. Optional full controlled Maze experiment

The previous notebook used `max_iter=16` for both training and evaluation. That made strict evaluation fail when 30 of 32 examples converged but two examples needed more iterations. This version uses:

- training ceiling: `24`
- evaluation ceiling: `64`
- fixed-point tolerance: `1e-4`
- strict convergence: still enabled
- stability power steps: `4`

The evaluation ceiling is not the charged cost. Anderson stops early, and the result records realized mean, p90, p99, and maximum iteration counts.


In [ ]:
def run_streaming(command):
    print("\n$", shlex.join(command), flush=True)
    subprocess.run(command, check=True)


if RUN_FULL_BENCHMARK:
    output_dir = Path(
        f"results/controlled_colab_t{TRAIN_MAX_ITER}_e{EVAL_MAX_ITER}"
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    shared = [
        "--task", "maze",
        "--maze_size", "7",
        "--extra_sizes", "9", "11",
        "--device", "cuda" if torch.cuda.is_available() else "cpu",
        "--steps", str(FULL_STEPS),
        "--batch_size", "32",
        "--lr", "3e-3",
        "--hidden", "128",
        "--heads", "4",
        "--max_iter", str(TRAIN_MAX_ITER),
        "--max_iter_eval", str(EVAL_MAX_ITER),
        "--fp_tol", "1e-4",
        "--eval_every", "100",
    ]

    for seed in FULL_SEEDS:
        prime_output = output_dir / f"prime_maze7_s{seed}.json"
        block_output = output_dir / f"deq_maze7_s{seed}.json"

        prime_command = [
            sys.executable,
            "-m",
            "experiments.fpsa_prime.controlled_compare",
            "--family", "prime",
            "--arch", "fpsa_prime",
            *shared,
            "--seed", str(seed),
            "--stability_power_steps", "4",
            "--output", str(prime_output),
            "--save_model", str(prime_output.with_suffix(".pt")),
        ]
        block_command = [
            sys.executable,
            "-m",
            "experiments.fpsa_prime.controlled_compare",
            "--family", "block",
            "--arch", "deq_block",
            *shared,
            "--seed", str(seed),
            "--output", str(block_output),
            "--save_model", str(block_output.with_suffix(".pt")),
        ]

        run_streaming(prime_command)
        run_streaming(block_command)

    rows = []
    histories = []
    for path in sorted(output_dir.glob("*.json")):
        result = json.loads(path.read_text())
        final = result["final"]
        rows.append(
            {
                "family": result["family"],
                "arch": result["arch"],
                "seed": result["seed"],
                "params": result["params"],
                "maze7_em": final["exact_match"],
                "maze9_em": result["extra"]["size9"]["exact_match"],
                "maze11_em": result["extra"]["size11"]["exact_match"],
                "mean_nfe": final["mean_function_evals"],
                "p90_iterations": final.get("p90_sample_iterations"),
                "p99_iterations": final.get("p99_sample_iterations"),
                "max_iterations": final.get("max_sample_iterations"),
                "max_residual": final.get("max_residual"),
                "min_batch_converged_fraction": final.get(
                    "min_batch_converged_fraction"
                ),
                "elapsed_seconds": result["elapsed_seconds"],
            }
        )
        for record in result["history"]:
            histories.append(
                {
                    "family": result["family"],
                    "arch": result["arch"],
                    "seed": result["seed"],
                    **record,
                }
            )

    full_results = pd.DataFrame(rows)
    history_table = pd.DataFrame(histories)
    display(full_results)

    numeric_columns = [
        "maze7_em",
        "maze9_em",
        "maze11_em",
        "mean_nfe",
        "p90_iterations",
        "p99_iterations",
        "max_iterations",
        "max_residual",
        "elapsed_seconds",
    ]
    display(
        full_results.groupby(["family", "arch"])[numeric_columns]
        .agg(["mean", "std"])
    )

    # Accuracy across grid sizes.
    accuracy_long = full_results.melt(
        id_vars=["family", "arch", "seed"],
        value_vars=["maze7_em", "maze9_em", "maze11_em"],
        var_name="grid",
        value_name="exact_match",
    )
    grid_to_size = {
        "maze7_em": 7,
        "maze9_em": 9,
        "maze11_em": 11,
    }
    accuracy_long["grid_size"] = accuracy_long["grid"].map(grid_to_size)

    plt.figure(figsize=(7.5, 4.5))
    for (family, arch), group in accuracy_long.groupby(
        ["family", "arch"]
    ):
        aggregate = (
            group.groupby("grid_size").exact_match
            .agg(["mean", "std"])
            .reset_index()
        )
        plt.errorbar(
            aggregate.grid_size,
            aggregate["mean"],
            yerr=aggregate["std"].fillna(0.0),
            marker="o",
            capsize=4,
            label=f"{family}/{arch}",
        )
    plt.xlabel("Maze grid size")
    plt.ylabel("Exact match (%)")
    plt.title("Controlled Maze accuracy and extrapolation")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Training trajectory.
    plt.figure(figsize=(7.5, 4.5))
    for (family, arch, seed), group in history_table.groupby(
        ["family", "arch", "seed"]
    ):
        plt.plot(
            group.step,
            group.exact_match,
            marker="o",
            label=f"{family}/{arch}/s{seed}",
        )
    plt.xlabel("Training step")
    plt.ylabel("Maze-7 exact match (%)")
    plt.title("Controlled training trajectory")
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print(
        "Full benchmark disabled. Set RUN_FULL_BENCHMARK=True after the "
        "verification cells pass."
    )


## Interpretation rules

- A cap of 64 does **not** mean 64 iterations are always executed. Use `mean_nfe`, `p90_iterations`, `p99_iterations`, and `max_iterations`.
- `min_batch_converged_fraction` must remain `1.0` for a strict equilibrium result.
- Do not use `--allow_nonconvergence` for the reported comparison.
- A failure even at 64 iterations is a real convergence failure and should be diagnosed, not hidden by loosening the tolerance.
- Start with one seed. Run three seeds only after forward and adjoint residuals remain clean for the full 1,200 steps.
